In [5]:
from bs4 import BeautifulSoup
import os
import time
import requests
from urllib.parse import urlparse

In [6]:
# --- CONFIGURATION ---
TEAM_URL = "https://www.basketmaniacs.com/team/unleash-the-clowns/"

# 1. Extract Team Name from URL (e.g., 'unleash-the-clowns')
parsed_url = urlparse(TEAM_URL)
team_slug = parsed_url.path.strip('/').split('/')[-1]
print(f" Target Team: {team_slug}")

# 2. Create the specific folder
base_dir = os.path.dirname(os.getcwd())
print(f"Base Directory: {base_dir}")
raw_data_dir = os.path.join(base_dir, 'data', 'raw', team_slug)

os.makedirs(raw_data_dir, exist_ok=True)

print(f"Folder ready: {raw_data_dir}")

 Target Team: unleash-the-clowns
Base Directory: /Users/nikos/basketball-stats
Folder ready: /Users/nikos/basketball-stats/data/raw/unleash-the-clowns


In [ ]:
# EXTRACT PLAYER LINKS 
response = requests.get(TEAM_URL)
soup = BeautifulSoup(response.text, 'html.parser')

# Find all player cards
roster_items = soup.find_all('div', class_='team-roster__item')

players_to_download = []

print(f"Scanning roster... Found {len(roster_items)} cards.\n")

for item in roster_items:
    # 1. Find the Link (<a> tag)
    link_tag = item.find('a', href=True)
    if not link_tag:
        continue # Skip if no link found (e.g., empty card)
        
    player_url = link_tag['href']
    
    # 2. Find the Name (<h2> tag)
    name_tag = item.find('h2', class_='team-roster__member-name')
    
    if name_tag:
        # Clean the name: Remove newlines, extra spaces
        raw_name = name_tag.get_text(separator=' ', strip=True)
        clean_name = " ".join(raw_name.split())
    else:
        # Fallback if name is missing
        clean_name = "Unknown_Player_" + player_url.split('/')[-1]

    # Add to our list
    players_to_download.append({
        'name': clean_name,
        'url': player_url
    })
    
    print(f"  Found: {clean_name}")

print(f"\n Ready to download {len(players_to_download)} players.")

Scanning roster... Found 19 cards.

   👤 Found: Παναγιώτης Ζέρβας
   👤 Found: Απόλλων Γραμματικόπουλος
   👤 Found: Pavlos Glynos
   👤 Found: Στεφανος Μεγαλος
   👤 Found: Ηλίας Χαλκίτης
   👤 Found: Θεόδωρος Κάγκας
   👤 Found: Νίκος Λάμπρου
   👤 Found: Φραγκισκος Βαμβακαρης
   👤 Found: Νικόλαος Βασσάκης
   👤 Found: Γιαννης Μαγειροπουλος
   👤 Found: Νότης Φακλης
   👤 Found: Σταύρος Παπαδόπουλος
   👤 Found: Νικηφορος Ελευθεριαδης
   👤 Found: Panagiotis Sourgias
   👤 Found: Αλέξης Δάφνης
   👤 Found: Σπυριδων Γρηγοριος Παπαδατος
   👤 Found: Christos Karametos
   👤 Found: Stelios Soulas
   👤 Found: Γιωργος Μουσετης

✅ Ready to download 19 players.


In [ ]:
import time
import re
import random


# 1. The "Mask" (Pretend to be a Browser)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def download_player(player_data):
    name = player_data['name']
    url = player_data['url']
    
    # Create clean filename (e.g., "Nikos_Vassakis.html")
    # This regex keeps Greek letters, English letters, and numbers
    safe_name = re.sub(r'[^\w\u0370-\u03FF_-]', '', name.replace(" ", "_"))
    filename = f"{safe_name}.html"
    save_path = os.path.join(raw_data_dir, filename)
    
    # Skip if we already have it
    if os.path.exists(save_path):
        print(f"⏩ Skipping {name} (Already downloaded)")
        return

    print(f"⬇️  Downloading: {name}...", end=" ")
    
    try:
        # Pass the HEADERS here
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status() # Check for errors (404, 500)
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print("✅ Done!")
        
        # Pause for a random time between 3-6 seconds to be polite to the server
        sleep_time = random.uniform(3, 6)
        print(f"   zzz... Sleeping for {sleep_time:.2f}s...")
        time.sleep(sleep_time)
        
    except Exception as e:
        print(f" Error: {e}")

# --- EXECUTE ---
print(f"Starting download for {len(players_to_download)} players...")
print("-" * 40)

for player in players_to_download:
    download_player(player)

print("-" * 40)
print(f"Success! All files are in: {raw_data_dir}")

Starting batch download for 19 players...
----------------------------------------
⏩ Skipping Παναγιώτης Ζέρβας (Already downloaded)
⬇️  Downloading: Απόλλων Γραμματικόπουλος... ✅ Done!
   zzz... Sleeping for 5.70s...
⬇️  Downloading: Pavlos Glynos... ✅ Done!
   zzz... Sleeping for 4.95s...
⬇️  Downloading: Στεφανος Μεγαλος... ✅ Done!
   zzz... Sleeping for 5.79s...
⬇️  Downloading: Ηλίας Χαλκίτης... ✅ Done!
   zzz... Sleeping for 5.10s...
⬇️  Downloading: Θεόδωρος Κάγκας... ✅ Done!
   zzz... Sleeping for 3.68s...
⬇️  Downloading: Νίκος Λάμπρου... ✅ Done!
   zzz... Sleeping for 5.39s...
⬇️  Downloading: Φραγκισκος Βαμβακαρης... ✅ Done!
   zzz... Sleeping for 3.14s...
⬇️  Downloading: Νικόλαος Βασσάκης... ✅ Done!
   zzz... Sleeping for 3.99s...
⬇️  Downloading: Γιαννης Μαγειροπουλος... ✅ Done!
   zzz... Sleeping for 5.94s...
⬇️  Downloading: Νότης Φακλης... ✅ Done!
   zzz... Sleeping for 3.78s...
⬇️  Downloading: Σταύρος Παπαδόπουλος... ✅ Done!
   zzz... Sleeping for 3.37s...
⬇️  Downlo